# 00 - Arranque

Ejecutar las celdas en orden en un servidor de Colab (CPU, RAM alta). Ver `README.md`.

In [1]:
import os
import sys

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "03DataCollectionAndEnvironmentSetup"

if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo pull

%cd /content/repo
!pip install -q -r requirements.txt
sys.path.insert(0, "/content/repo/src")

Already up to date.
/content/repo


In [3]:
from ptcg_wp.colab import setup_kaggle

setup_kaggle()
!kaggle datasets list -s "pokemon tcg ai battle" | head -5

ref                                               title                                                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------  -----------------------------------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
busyaprime/pokemon-tcg-ai-battle-live-meta        Pokemon TCG AI Battle: live ladder meta                                              2614  2026-08-01 16:00:59.037000            858         13  0.64705884       
kaggle/pokemon-tcg-ai-battle-episodes-index       The Pokémon Company - PTCG AI Battle Challenge Simulation Episodes Index             2606  2026-08-31 00:13:57.020000         104375         36  0.4117647        
citerne/pokemon-tcg-agent-showcase-media          Pokemon TCG Agent Showcase Media                                                 29061831  2026-08

## Ingesta

Descarga cada día de `configs/dias.yaml`, escribe un Parquet por día y borra el crudo.

In [ ]:
!PYTHONPATH=src python -m ptcg_wp.ingest --config configs/dias.yaml --raw /content/raw --out /content/datos

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.concat(pd.read_parquet(p) for p in sorted(Path("/content/datos").glob("partidas_*.parquet")))
print(df.shape)
print(df["winner"].value_counts())
df.head()

In [4]:
from ptcg_wp.ingest import download_day
import glob, json

raw = download_day("2026-06-18", "/content/raw")
files = sorted(glob.glob(f"{raw}/*.json"))
g = json.load(open(files[0]))

paso = g["steps"][50]
actuando = next(j for j in (0, 1) if paso[j]["status"] == "ACTIVE")
cur = paso[actuando]["observation"]["current"]

print("claves de current:", list(cur.keys()))
print("claves de player:", list(cur["players"][0].keys()))

for k, p in enumerate(cur["players"]):
    print(f"\n=== jugador {k} | active[0] ===")
    print(json.dumps(p["active"][0], indent=1, ensure_ascii=False))
    if p["bench"]:
        print(f"\n=== jugador {k} | bench[0] ===")
        print(json.dumps(p["bench"][0], indent=1, ensure_ascii=False))

claves de current: ['energyAttached', 'firstPlayer', 'looking', 'players', 'result', 'retreated', 'stadium', 'stadiumPlayed', 'supporterPlayed', 'turn', 'turnActionCount', 'yourIndex']
claves de player: ['active', 'asleep', 'bench', 'benchMax', 'burned', 'confused', 'deckCount', 'discard', 'hand', 'handCount', 'paralyzed', 'poisoned', 'prize']

=== jugador 0 | active[0] ===
{
 "appearThisTurn": false,
 "energies": [
  6
 ],
 "energyCards": [
  {
   "id": 6,
   "playerIndex": 0,
   "serial": 56
  }
 ],
 "hp": 340,
 "id": 678,
 "maxHp": 340,
 "playerIndex": 0,
 "preEvolution": [
  {
   "id": 677,
   "playerIndex": 0,
   "serial": 12
  }
 ],
 "serial": 18,
 "tools": []
}

=== jugador 0 | bench[0] ===
{
 "appearThisTurn": false,
 "energies": [
  6,
  6
 ],
 "energyCards": [
  {
   "id": 6,
   "playerIndex": 0,
   "serial": 53
  },
  {
   "id": 6,
   "playerIndex": 0,
   "serial": 59
  }
 ],
 "hp": 150,
 "id": 674,
 "maxHp": 150,
 "playerIndex": 0,
 "preEvolution": [
  {
   "id": 673,
   "p